# ============================================================
# AIMLZC416 – Mathematical Foundations for Machine Learning
# Assignment I  |  I Semester 2026-27
# ============================================================

## Name: Ayushi Pradhan
## BITS ID: 2026AJ05077

In [39]:
import numpy as np
import time
np.set_printoptions(precision=6, suppress=True)

## Q1: Finding Solutions of Linear Systems

In [40]:
# ----------------------------------------------------------------------
# Utility: generate a random float of the form n.dddddddd (8 decimals)
# ----------------------------------------------------------------------
def random_decimal(low=-9, high=9, rng=None):
    """Return a single random float of the form n.dddddddd (8 decimal digits)."""
    if rng is None:
        rng = np.random.default_rng()
    val = rng.uniform(low, high)
    return round(val, 8)


def random_matrix(m, n, low=-9, high=9, seed=None):
    """Generate an m x n matrix whose entries are of the form n.dddddddd."""
    rng = np.random.default_rng(seed)
    A = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            A[i, j] = random_decimal(low, high, rng)
    return A


def random_vector(m, low=-9, high=9, seed=None):
    rng = np.random.default_rng(seed)
    b = np.zeros((m, 1))
    for i in range(m):
        b[i, 0] = random_decimal(low, high, rng)
    return b

----------------------------------------------------------------------
1.1.a. REF from scratch (Gaussian elimination)

----------------------------------------------------------------------

In [51]:
def compute_ref(Ab, tol=1e-10):
    """
    Compute the Row Echelon Form (REF) of an augmented matrix Ab (m x (n+1))
    using Gaussian elimination with partial pivoting
    """
    M = Ab.astype(float).copy()
    m, ncols = M.shape
    pivot_row = 0
    pivot_cols = []

    for col in range(ncols - 1):  # last column is b, don't pivot on it
        if pivot_row >= m:
            break

        # --- partial pivoting: find the row (>= pivot_row) with the
        # largest absolute value in this column, to avoid division by
        # (near) zero ---
        max_val = abs(M[pivot_row, col])
        max_row = pivot_row
        for r in range(pivot_row + 1, m):
            if abs(M[r, col]) > max_val:
                max_val = abs(M[r, col])
                max_row = r

        # if the best pivot candidate is (numerically) zero, this column
        # has no pivot -> move to next column
        if max_val < tol:
            continue

        # swap rows manually
        if max_row != pivot_row:
            temp = M[pivot_row, :].copy()
            M[pivot_row, :] = M[max_row, :]
            M[max_row, :] = temp

        # eliminate all entries BELOW the pivot (REF only needs below)
        for r in range(pivot_row + 1, m):
            factor = M[r, col] / M[pivot_row, col]
            M[r, :] = M[r, :] - factor * M[pivot_row, :]

        pivot_cols.append(col)
        pivot_row += 1

    return M, pivot_cols

----------------------------------------------------------------------
1.1.b. RREF from scratch 

----------------------------------------------------------------------

In [53]:
def compute_rref(REF, pivot_cols, tol=1e-10):
    """
    Given a REF matrix and its pivot columns, reduce it further to RREF:
    make every pivot equal to 1 and eliminate ABOVE each pivot as well.
    """
    M = REF.astype(float).copy()

    # normalise each pivot row so the pivot entry becomes exactly 1
    for i, col in enumerate(pivot_cols):
        pivot_val = M[i, col]
        M[i, :] = M[i, :] / pivot_val

        # eliminate this column in every OTHER row (above and below)
        for r in range(M.shape[0]):
            if r != i and abs(M[r, col]) > tol:
                factor = M[r, col]
                M[r, :] = M[r, :] - factor * M[i, :]

    return M

----------------------------------------------------------------------

1.2. Pivot / non-pivot columns, particular solution, null-space solutions

----------------------------------------------------------------------

In [43]:
def pivot_and_free_columns(RREF, pivot_cols, n):
    """n = number of columns of A (i.e. RREF has n+1 columns: A | b)."""
    all_cols = list(range(n))
    free_cols = [c for c in all_cols if c not in pivot_cols]
    return pivot_cols, free_cols


def particular_solution(RREF, pivot_cols, free_cols, n):
    """
    Particular solution: set all free variables to 0, read off pivot
    variables from the RREF's last (b) column.
    """
    xp = np.zeros((n, 1))
    for i, col in enumerate(pivot_cols):
        xp[col, 0] = RREF[i, -1]
    return xp


def nullspace_solutions(RREF, pivot_cols, free_cols, n):
    """
    Solutions of Ax = 0 (a basis for the null space): for every free
    variable, set it to 1, all other free variables to 0, and back out
    the pivot variables using the RREF rows.
    """
    basis = []
    for free_col in free_cols:
        x = np.zeros((n, 1))
        x[free_col, 0] = 1.0
        for i, pcol in enumerate(pivot_cols):
            # RREF row i:  x[pcol] + sum_{free f} RREF[i,f]*x[f] = 0
            x[pcol, 0] = -RREF[i, free_col]
        basis.append(x)
    return basis


def general_solution_str(free_cols):
    terms = " + ".join([f"t{k+1}*v{k+1}" for k in range(len(free_cols))])
    return f"x = xp" + (f" + {terms}" if terms else "")


def verify_solution(A, b, x, tol=1e-6):
    residual = A @ x - b
    return residual, np.max(np.abs(residual)) < tol

----------------------------------------------------------------------
1.3. Demo to show the REF, RREF, pivot columns, non-pivot columns, the particular solution, the solutions to Ax = 0, the general solution and verify the general solution.

----------------------------------------------------------------------

In [55]:
print("="*70)
print("Q1.3  Demonstration on a random 5x7 matrix A and a suitable b")
print("="*70)

m, n = 5, 7
seed = 42
A = random_matrix(m, n, seed=seed)

# "suitable b": we need m < n (5<7, satisfied) and we want the system to be
# consistent, so we build b = A @ x_true for some random true x, guaranteeing
# a solution exists (rather than an inconsistent random b).
rng = np.random.default_rng(123)
x_true = np.array([[random_decimal(-9, 9, rng)] for _ in range(n)])
b = A @ x_true

print("\nMatrix A (5x7):\n", A)
print("\nVector b (5x1) [built as A @ x_true so the system is consistent]:\n", b)

Ab = np.hstack([A, b])
print("\nAugmented matrix [A | b]:\n", Ab)

REF, pivot_cols = compute_ref(Ab)
print("\n--- REF ---\n", REF)

RREF = compute_rref(REF, pivot_cols)
print("\n--- RREF ---\n", RREF)

pivot_cols, free_cols = pivot_and_free_columns(RREF, pivot_cols, n)
print("\nPivot columns (0-indexed):", pivot_cols)
print("Non-pivot (free) columns (0-indexed):", free_cols)

xp = particular_solution(RREF, pivot_cols, free_cols, n)
print("\nParticular solution x_p:\n", xp)

null_basis = nullspace_solutions(RREF, pivot_cols, free_cols, n)
print("\nBasis for solutions of Ax = 0 (null space), one vector per free column:")
for i, v in enumerate(null_basis):
    print(f"v{i+1} =\n{v}")

print("\nGeneral solution:", general_solution_str(free_cols))
print("  where t1, t2, ... are free scalar parameters")

# Verification: particular solution should satisfy A xp = b
res_p, ok_p = verify_solution(A, b, xp)
print("\nVerify A@xp - b (should be ~0):\n", res_p.T, " -> consistent:", ok_p)

# Verification: each null-space vector should satisfy A v = 0
for i, v in enumerate(null_basis):
    res, ok = verify_solution(A, np.zeros((m,1)), v)
    print(f"Verify A@v{i+1} (should be ~0):", res.T.round(6), " -> in null space:", ok)

# Verification: general solution for random t's also satisfies Ax=b
if len(null_basis) > 0:
    ts = [random_decimal(-5,5) for _ in null_basis]
    x_general = xp.copy()
    for t, v in zip(ts, null_basis):
        x_general = x_general + t * v
    res_g, ok_g = verify_solution(A, b, x_general)
    print("\nRandom t's used:", ts)
    print("General solution x = xp + sum(t_i * v_i):\n", x_general)
    print("Verify A@x_general - b (should be ~0):\n", res_g.T, " -> consistent:", ok_g)

Q1.3  Demonstration on a random 5x7 matrix A and a suitable b

Matrix A (5x7):
 [[ 4.931209 -1.100188  6.454763  3.552625 -7.304808  8.561202  4.700515]
 [ 5.149157 -6.693955 -0.893053 -2.325636  7.68177   2.589572  5.809709]
 [-1.018544 -4.909703  0.982526 -7.851289  5.897361  2.369959  4.645579]
 [-2.618533  8.472564  7.07618   5.010903 -5.496503 -0.599022 -8.211532]
 [-6.222789  3.294881  4.405719  8.415175 -3.135144 -2.331725 -0.547995]]

Vector b (5x1) [built as A @ x_true so the system is consistent]:
 [[  98.875179]
 [ 102.375291]
 [  90.058345]
 [-174.60034 ]
 [-115.85798 ]]

Augmented matrix [A | b]:
 [[   4.931209   -1.100188    6.454763    3.552625   -7.304808    8.561202
     4.700515   98.875179]
 [   5.149157   -6.693955   -0.893053   -2.325636    7.68177     2.589572
     5.809709  102.375291]
 [  -1.018544   -4.909703    0.982526   -7.851289    5.897361    2.369959
     4.645579   90.058345]
 [  -2.618533    8.472564    7.07618     5.010903   -5.496503   -0.599022
    -

## Q2: Dataset Generation & Rank via RREF

----------------------------------------------------------------------
2.1. Generate the dataset X (500 x 6)

----------------------------------------------------------------------

In [56]:
def generate_dataset(n_samples=500, seed=1):
    rng = np.random.default_rng(seed)
    f1 = rng.standard_normal(n_samples)   # standard normal N(0,1)
    f2 = rng.standard_normal(n_samples)
    f3 = rng.standard_normal(n_samples)
    f4 = rng.standard_normal(n_samples)
    f5 = 2 * f1 + 3 * f2                  # linear combination of f1, f2
    f6 = f3 - 2 * f4                      # linear combination of f3, f4

    X = np.column_stack([f1, f2, f3, f4, f5, f6])
    return X


----------------------------------------------------------------------
2.2. Rank of X, computed manually via REF

----------------------------------------------------------------------

In [57]:
def compute_rank(X, tol=1e-8):
    # Treat X itself as the coefficient part; append a dummy zero column
    # so we can reuse compute_ref (which expects an augmented matrix).
    dummy_b = np.zeros((X.shape[0], 1))
    Xb = np.hstack([X, dummy_b])
    REF, pivot_cols = compute_ref(Xb, tol=tol)
    return len(pivot_cols), REF

----------------------------------------------------------------------
Demo to compute the rank of X and display the output for the dataset
X generated 

----------------------------------------------------------------------

In [58]:
print("="*70)
print("Q2.1  Generating dataset X (500 x 6)")
print("="*70)
X = generate_dataset(n_samples=500, seed=1)
print("X.shape =", X.shape)
print("First 5 rows of X:\n", X[:5])

print("\n" + "="*70)
print("Q2.2  Rank of X (computed manually via REF / Gaussian elimination)")
print("="*70)
rank, REF = compute_rank(X)
print("Rank of X =", rank, " (expected 4, since f5,f6 are linear combinations of f1..f4)")

Q2.1  Generating dataset X (500 x 6)
X.shape = (500, 6)
First 5 rows of X:
 [[ 0.345584 -1.37034   0.19484   0.388527 -3.419852 -0.582214]
 [ 0.821618  2.175598  0.838718  1.26683   8.17003  -1.694941]
 [ 0.330437 -1.387413 -0.027649  2.166993 -3.501366 -4.361635]
 [-1.303157 -1.07752   0.782432  2.848904 -5.838876 -4.915376]
 [ 0.905356 -1.200863 -2.575443  1.350835 -1.791878 -5.277112]]

Q2.2  Rank of X (computed manually via REF / Gaussian elimination)
Rank of X = 4  (expected 4, since f5,f6 are linear combinations of f1..f4)


## Q2.3: Power Method & Deflation

----------------------------------------------------------------------
2.3(a). Covariance matrix C = (1/n) X^T X

----------------------------------------------------------------------

In [59]:
def covariance_matrix(X):
    n = X.shape[0]
    C = np.zeros((X.shape[1], X.shape[1]))
    # manual matrix multiplication X^T X, then scale by 1/n
    Xt = X.T
    for i in range(Xt.shape[0]):
        for j in range(X.shape[1]):
            C[i, j] = np.sum(Xt[i, :] * X[:, j])
    C = C / n
    return C


----------------------------------------------------------------------
2.3(b). Power Method for dominant eigenvalue/eigenvector

----------------------------------------------------------------------

In [60]:
def power_method(C, tol=1e-7, max_iter=100000, seed=0):
    rng = np.random.default_rng(seed)
    d = C.shape[0]
    v = rng.standard_normal(d)
    v = v / np.sqrt(np.sum(v ** 2))   # normalise manually (no np.linalg.norm)

    lam_old = 0.0
    for it in range(1, max_iter + 1):
        w = C @ v
        norm_w = np.sqrt(np.sum(w ** 2))
        v_new = w / norm_w
        # Rayleigh quotient estimate of the eigenvalue
        lam_new = v_new @ (C @ v_new)

        if abs(lam_new - lam_old) < tol:
            v = v_new
            lam_old = lam_new
            break
        v = v_new
        lam_old = lam_new

    # fix sign convention: make largest-magnitude component positive
    if v[np.argmax(np.abs(v))] < 0:
        v = -v
    return lam_old, v, it

----------------------------------------------------------------------
2.3(c). Deflation: get next eigenvalue/vector from C - v v^T C

----------------------------------------------------------------------

In [61]:
def deflate(C, eigvecs):
    """Subtract the effect of already-found eigenvectors from C."""
    C_def = C.copy()
    for v in eigvecs:
        v = v.reshape(-1, 1)
        C_def = C_def - v @ (v.T @ C)
    return C_def


def all_eigen_via_power_method(C, n_components, tol=1e-7, seed=0):
    eigvals = []
    eigvecs = []
    iters = []
    C_cur = C.copy()
    for k in range(n_components):
        lam, v, it = power_method(C_cur, tol=tol, seed=seed + k)
        eigvals.append(lam)
        eigvecs.append(v)
        iters.append(it)
        C_cur = deflate(C, eigvecs)   # deflate original C by all vectors found so far
    return eigvals, eigvecs, iters

In [62]:
print("\n" + "="*70)
print("Q2.3(a)  Covariance matrix C = (1/n) X^T X")
print("="*70)
C = covariance_matrix(X)
print("C =\n", C)

print("\n" + "="*70)
print("Q2.3(b)  Power Method: dominant eigenvalue/eigenvector")
print("="*70)
lam1, v1, it1 = power_method(C, tol=1e-7, seed=0)
print(f"lambda_1 = {lam1:.8f}")
print("v1 =\n", v1)
print("iterations used:", it1)

print("\n" + "="*70)
print("Q2.3(c)  Next eigenvalues/eigenvectors via deflation")
print("="*70)
eigvals, eigvecs, iters = all_eigen_via_power_method(C, n_components=6, tol=1e-7, seed=0)
for k in range(6):
    print(f"\nlambda_{k+1} = {eigvals[k]:.8f}  (iterations: {iters[k]})")
    print(f"v_{k+1} =\n", eigvecs[k])

print("\n" + "="*70)
print("Q2.3(d)  Eigenvalues/eigenvectors via built-in Python function (np.linalg.eigh)")
print("="*70)
eigvals_np, eigvecs_np = np.linalg.eigh(C)   # symmetric matrix -> eigh
# sort descending to match power-method ordering
order = np.argsort(eigvals_np)[::-1]
eigvals_np = eigvals_np[order]
eigvecs_np = eigvecs_np[:, order]

print("Eigenvalues (descending):", eigvals_np)
print("\nComparison table (Power method vs np.linalg.eigh):")
print(f"{'k':<3}{'PowerMethod':<15}{'np.linalg.eigh':<15}{'abs diff':<12}")
for k in range(6):
    diff = abs(eigvals[k] - eigvals_np[k])
    print(f"{k+1:<3}{eigvals[k]:<15.8f}{eigvals_np[k]:<15.8f}{diff:<12.2e}")

print("\nEigenvector comparison (dot product magnitude, should be ~1 if aligned):")
for k in range(6):
    v_pm = eigvecs[k] / np.sqrt(np.sum(eigvecs[k]**2))
    v_np = eigvecs_np[:, k]
    dot = abs(np.sum(v_pm * v_np))
    print(f"k={k+1}: |v_pm . v_np| = {dot:.6f}")

print("\n" + "="*70)
print("Q2.3(e)  Iterations to reach 1e-7 accuracy: power method vs 'exact' (eigh) values")
print("="*70)

def power_method_track_iters(C, true_lambda, tol=1e-7, max_iter=200000, seed=0):
    rng = np.random.default_rng(seed)
    d = C.shape[0]
    v = rng.standard_normal(d)
    v = v / np.sqrt(np.sum(v**2))
    for it in range(1, max_iter+1):
        w = C @ v
        norm_w = np.sqrt(np.sum(w**2))
        v = w / norm_w
        lam = v @ (C @ v)
        if abs(lam - true_lambda) < tol:
            return it, lam
    return max_iter, lam

print(f"{'k':<3}{'true_lambda(eigh)':<20}{'iters_to_1e-7':<15}")
for k in range(6):
    C_def = deflate(C, eigvecs[:k]) if k > 0 else C
    it_needed, lam_final = power_method_track_iters(C_def, eigvals_np[k], tol=1e-7, seed=k)
    print(f"{k+1:<3}{eigvals_np[k]:<20.8f}{it_needed:<15}")


Q2.3(a)  Covariance matrix C = (1/n) X^T X
C =
 [[ 0.834756  0.002171  0.032589  0.050246  1.676026 -0.067903]
 [ 0.002171  1.116553  0.099004  0.050338  3.354    -0.001671]
 [ 0.032589  0.099004  1.102168 -0.014081  0.362191  1.13033 ]
 [ 0.050246  0.050338 -0.014081  1.00019   0.251505 -2.014461]
 [ 1.676026  3.354     0.362191  0.251505 13.414051 -0.140818]
 [-0.067903 -0.001671  1.13033  -2.014461 -0.140818  5.159251]]

Q2.3(b)  Power Method: dominant eigenvalue/eigenvector
lambda_1 = 14.48133488
v1 =
 [ 0.118454  0.241799  0.02667   0.021808  0.962304 -0.016945]
iterations used: 12

Q2.3(c)  Next eigenvalues/eigenvectors via deflation

lambda_1 = 14.48133488  (iterations: 12)
v_1 =
 [ 0.118454  0.241799  0.02667   0.021808  0.962304 -0.016945]

lambda_2 = 6.19217606  (iterations: 6)
v_2 =
 [-0.008432  0.011167  0.204924 -0.353722  0.016635  0.912368]

lambda_3 = 1.05210350  (iterations: 36)
v_3 =
 [-0.009844 -0.00445   0.88913   0.455782 -0.033036 -0.022434]

lambda_4 = 0.9013545